In [ ]:
import pandas as pd

In [ ]:
df= pd.read_csv(r"D:\projects\telco customer churn\dataset\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.columns

In [ ]:
print(df.shape)


In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['TotalCharges'].unique()

# DATA PREPARATION

#Total Charges dtype->string -> convert to numeric

In [ ]:
df['TotalCharges'].dtype

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
print(df[df["TotalCharges"].isna()][["tenure", "TotalCharges"]])  # confirm tenure is 0

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
df['TotalCharges'].isnull().sum()

#Drop customer id and encoding the target

In [ ]:
df= df.drop(columns="customerID")

In [ ]:
df["Churn"] = (df["Churn"] == "Yes").astype(int)

In [ ]:
print(df.isna().sum().sum(), "missing values left")
print(df["Churn"].mean())  

# EDA

#

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    print(df.groupby(col)["Churn"].mean().sort_values(ascending=False).round(3), "\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, ax=axes[0])
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
plt.show()

# FEATURE ENGINEERING

In [ ]:
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)
df["charges_per_month"] = df["TotalCharges"] / (df["tenure"] + 1)
df["tenure_bucket"] = pd.cut(df["tenure"], [-1, 12, 24, 48, 72],
                             labels=["0-12", "13-24", "25-48", "49-72"]).astype(str)

# MODEL BUILDING

#data split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# processing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

#Comparing models with cross validation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "logreg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "xgb": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                         scale_pos_weight=pos_weight, eval_metric="logloss", random_state=42),
    "lgbm": LGBMClassifier(n_estimators=300, learning_rate=0.05,
                           class_weight="balanced", random_state=42, verbose=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"{name:7s} PR-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")

#Choosing threshold by business cost

In [ ]:
offer_cost = 500        # cost of a retention offer per customer contacted
customer_value = 5000   # value of a retained customer
save_rate = 0.30        # share of true churners who accept the offer

thresholds = np.linspace(0.05, 0.95, 91)
profits = []
for t in thresholds:
    pred = proba >= t
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    profits.append(tp * (customer_value * save_rate - offer_cost) - fp * offer_cost)

best_t = thresholds[int(np.argmax(profits))]
print("Best threshold:", round(best_t, 2), "| Profit:", max(profits))

plt.plot(thresholds, profits); plt.xlabel("Threshold"); plt.ylabel("Expected profit"); plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)

best = Pipeline([("pre", pre), ("model", models["xgb"])])
best.fit(X_train, y_train)

proba = best.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC :", round(average_precision_score(y_test, proba), 3))
print(classification_report(y_test, (proba >= 0.5).astype(int)))

#Explaining model using shap

In [ ]:
import shap

Xt = best.named_steps["pre"].transform(X_test)
names = best.named_steps["pre"].get_feature_names_out()

explainer = shap.TreeExplainer(best.named_steps["model"])
shap_values = explainer.shap_values(Xt)

shap.summary_plot(shap_values, Xt, feature_names=names)

#storing model in joblit for streamlit

In [ ]:
import joblib
joblib.dump(best, r"D:\projects\telco customer churn\code files\churn_pipeline.joblib")